
Transformer Inference | How Inference is done in Transformer

In [1]:
"""
=============================================================
  Transformer Inference — Complete Implementation from Scratch
=============================================================
  Covers:
    1. Scaled Dot-Product Attention
    2. Multi-Head Attention
    3. Feed-Forward Network
    4. Transformer Layer (Encoder & Decoder)
    5. Full Encoder-Decoder Transformer
    6. Autoregressive Decoding (Greedy, Top-K, Top-P)
    7. KV Cache for efficient inference
=============================================================
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Tuple, List


# ─────────────────────────────────────────────────────────────
# 1. Scaled Dot-Product Attention
# ─────────────────────────────────────────────────────────────

def scaled_dot_product_attention(
    Q: torch.Tensor,            # [batch, heads, seq_q, d_k]
    K: torch.Tensor,            # [batch, heads, seq_k, d_k]
    V: torch.Tensor,            # [batch, heads, seq_k, d_v]
    mask: Optional[torch.Tensor] = None   # [batch, 1, seq_q, seq_k]
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) * V
    Returns: output [batch, heads, seq_q, d_v] and attention weights
    """
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)  # [b, h, seq_q, seq_k]

    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    attn_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attn_weights, V)
    return output, attn_weights


# ─────────────────────────────────────────────────────────────
# 2. Multi-Head Attention
# ─────────────────────────────────────────────────────────────

class MultiHeadAttention(nn.Module):
    """
    Splits Q, K, V into h heads, runs attention in parallel,
    then concatenates and projects back.
    """

    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)  # Query projection
        self.W_k = nn.Linear(d_model, d_model)  # Key projection
        self.W_v = nn.Linear(d_model, d_model)  # Value projection
        self.W_o = nn.Linear(d_model, d_model)  # Output projection

    def split_heads(self, x: torch.Tensor) -> torch.Tensor:
        """[batch, seq, d_model] → [batch, heads, seq, d_k]"""
        B, T, _ = x.shape
        x = x.view(B, T, self.num_heads, self.d_k)
        return x.transpose(1, 2)

    def merge_heads(self, x: torch.Tensor) -> torch.Tensor:
        """[batch, heads, seq, d_k] → [batch, seq, d_model]"""
        B, _, T, _ = x.shape
        x = x.transpose(1, 2).contiguous()
        return x.view(B, T, self.d_model)

    def forward(
        self,
        query: torch.Tensor,
        key: torch.Tensor,
        value: torch.Tensor,
        mask: Optional[torch.Tensor] = None,
        kv_cache: Optional[Tuple[torch.Tensor, torch.Tensor]] = None
    ) -> Tuple[torch.Tensor, Optional[Tuple[torch.Tensor, torch.Tensor]]]:
        """
        Args:
            query, key, value: input tensors [batch, seq, d_model]
            mask: optional attention mask
            kv_cache: (cached_K, cached_V) from previous steps
        Returns:
            output [batch, seq, d_model], new kv_cache
        """
        Q = self.split_heads(self.W_q(query))
        K = self.split_heads(self.W_k(key))
        V = self.split_heads(self.W_v(value))

        # ── KV Cache: append new K, V to cached ──
        if kv_cache is not None:
            cached_K, cached_V = kv_cache
            K = torch.cat([cached_K, K], dim=2)
            V = torch.cat([cached_V, V], dim=2)

        new_kv_cache = (K, V)

        attn_out, _ = scaled_dot_product_attention(Q, K, V, mask)
        output = self.W_o(self.merge_heads(attn_out))
        return output, new_kv_cache


# ─────────────────────────────────────────────────────────────
# 3. Position-wise Feed-Forward Network
# ─────────────────────────────────────────────────────────────

class FeedForward(nn.Module):
    """
    Two-layer MLP applied independently to each position:
      FFN(x) = GELU(xW1 + b1)W2 + b2
    """

    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# ─────────────────────────────────────────────────────────────
# 4. Positional Encoding
# ─────────────────────────────────────────────────────────────

class PositionalEncoding(nn.Module):
    """
    Adds sinusoidal positional encodings to embeddings.
    PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """

    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(max_len).unsqueeze(1)  # [max_len, 1]
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))  # [1, max_len, d_model]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: [batch, seq, d_model]"""
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# ─────────────────────────────────────────────────────────────
# 5. Encoder Layer
# ─────────────────────────────────────────────────────────────

class EncoderLayer(nn.Module):
    """
    Single Transformer encoder layer:
      x = LayerNorm(x + SelfAttention(x))
      x = LayerNorm(x + FFN(x))
    """

    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, src_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        # Self-attention with residual
        attn_out, _ = self.self_attn(x, x, x, mask=src_mask)
        x = self.norm1(x + self.dropout(attn_out))
        # Feed-forward with residual
        x = self.norm2(x + self.dropout(self.ffn(x)))
        return x


# ─────────────────────────────────────────────────────────────
# 6. Decoder Layer
# ─────────────────────────────────────────────────────────────

class DecoderLayer(nn.Module):
    """
    Single Transformer decoder layer:
      x = LayerNorm(x + MaskedSelfAttention(x))       ← causal mask
      x = LayerNorm(x + CrossAttention(x, enc_out))   ← attends to encoder
      x = LayerNorm(x + FFN(x))
    """

    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        x: torch.Tensor,
        enc_out: torch.Tensor,
        tgt_mask: Optional[torch.Tensor] = None,
        src_mask: Optional[torch.Tensor] = None,
        self_kv_cache: Optional[Tuple] = None,
        cross_kv_cache: Optional[Tuple] = None,
    ) -> Tuple[torch.Tensor, Tuple, Tuple]:
        # 1) Masked self-attention
        attn_out, new_self_kv = self.self_attn(x, x, x, mask=tgt_mask, kv_cache=self_kv_cache)
        x = self.norm1(x + self.dropout(attn_out))

        # 2) Cross-attention (decoder queries attend to encoder keys/values)
        cross_out, new_cross_kv = self.cross_attn(x, enc_out, enc_out, mask=src_mask, kv_cache=cross_kv_cache)
        x = self.norm2(x + self.dropout(cross_out))

        # 3) Feed-forward
        x = self.norm3(x + self.dropout(self.ffn(x)))
        return x, new_self_kv, new_cross_kv


# ─────────────────────────────────────────────────────────────
# 7. Full Encoder-Decoder Transformer
# ─────────────────────────────────────────────────────────────

class Transformer(nn.Module):
    """
    Full sequence-to-sequence Transformer (T5 / BART style).
    For decoder-only (GPT style), just use the decoder with causal masking.
    """

    def __init__(
        self,
        vocab_size: int,
        d_model: int = 512,
        num_heads: int = 8,
        num_layers: int = 6,
        d_ff: int = 2048,
        max_len: int = 512,
        dropout: float = 0.1,
        pad_idx: int = 0,
    ):
        super().__init__()
        self.d_model = d_model
        self.pad_idx = pad_idx

        # Shared embedding (encoder and decoder)
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos_encoding = PositionalEncoding(d_model, max_len, dropout)

        # Encoder stack
        self.encoder_layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

        # Decoder stack
        self.decoder_layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

        # Output projection to vocabulary
        self.lm_head = nn.Linear(d_model, vocab_size)

        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def make_causal_mask(self, seq_len: int, device: torch.device) -> torch.Tensor:
        """Lower-triangular causal mask: position i can only attend to positions <= i"""
        mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
        return mask.unsqueeze(0).unsqueeze(0)   # [1, 1, seq, seq]

    def make_padding_mask(self, ids: torch.Tensor) -> torch.Tensor:
        """Mask out padding tokens"""
        return (ids != self.pad_idx).unsqueeze(1).unsqueeze(2)   # [B, 1, 1, seq]

    def encode(self, src_ids: torch.Tensor) -> torch.Tensor:
        """Encode source sequence → hidden states [B, src_len, d_model]"""
        src_mask = self.make_padding_mask(src_ids)
        x = self.pos_encoding(self.embedding(src_ids) * math.sqrt(self.d_model))
        for layer in self.encoder_layers:
            x = layer(x, src_mask)
        return x

    def decode_step(
        self,
        tgt_ids: torch.Tensor,
        enc_out: torch.Tensor,
        kv_caches: Optional[List] = None,
    ) -> Tuple[torch.Tensor, List]:
        """
        Single decoder forward pass with KV cache support.
        Returns logits for the last token and updated caches.
        """
        tgt_len = tgt_ids.size(1)
        tgt_mask = self.make_causal_mask(tgt_len, tgt_ids.device)

        x = self.pos_encoding(self.embedding(tgt_ids) * math.sqrt(self.d_model))

        new_caches = []
        for i, layer in enumerate(self.decoder_layers):
            self_kv = kv_caches[i][0] if kv_caches else None
            cross_kv = kv_caches[i][1] if kv_caches else None
            x, new_self_kv, new_cross_kv = layer(x, enc_out, tgt_mask=tgt_mask,
                                                   self_kv_cache=self_kv,
                                                   cross_kv_cache=cross_kv)
            new_caches.append((new_self_kv, new_cross_kv))

        logits = self.lm_head(x)   # [B, seq, vocab_size]
        return logits, new_caches


# ─────────────────────────────────────────────────────────────
# 8. Decoding Strategies
# ─────────────────────────────────────────────────────────────

def greedy_decode(logits: torch.Tensor) -> torch.Tensor:
    """Always pick the most probable token."""
    return logits[:, -1, :].argmax(dim=-1)


def top_k_sample(logits: torch.Tensor, k: int = 50, temperature: float = 1.0) -> torch.Tensor:
    """Sample from the top-K most probable tokens."""
    logits = logits[:, -1, :] / temperature
    top_k_logits, top_k_indices = torch.topk(logits, k, dim=-1)
    probs = F.softmax(top_k_logits, dim=-1)
    chosen = torch.multinomial(probs, num_samples=1)
    return top_k_indices.gather(-1, chosen).squeeze(-1)


def top_p_sample(logits: torch.Tensor, p: float = 0.9, temperature: float = 1.0) -> torch.Tensor:
    """
    Nucleus sampling: sample from the smallest set of tokens
    whose cumulative probability exceeds p.
    """
    logits = logits[:, -1, :] / temperature
    sorted_logits, sorted_indices = torch.sort(logits, descending=True, dim=-1)
    cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)

    # Remove tokens with cumulative probability above threshold
    sorted_indices_to_remove = cumulative_probs - F.softmax(sorted_logits, dim=-1) > p
    sorted_logits[sorted_indices_to_remove] = float('-inf')

    probs = F.softmax(sorted_logits, dim=-1)
    chosen_sorted = torch.multinomial(probs, num_samples=1)
    return sorted_indices.gather(-1, chosen_sorted).squeeze(-1)


# ─────────────────────────────────────────────────────────────
# 9. Full Inference Loop with KV Cache
# ─────────────────────────────────────────────────────────────

@torch.no_grad()
def generate(
    model: Transformer,
    src_ids: torch.Tensor,
    bos_token_id: int,
    eos_token_id: int,
    max_new_tokens: int = 100,
    decoding: str = "greedy",   # "greedy" | "top_k" | "top_p"
    temperature: float = 1.0,
    top_k: int = 50,
    top_p: float = 0.9,
    device: str = "cpu"
) -> List[int]:
    """
    Full autoregressive generation loop with:
      - Encoder runs once
      - Decoder uses KV cache for efficient token-by-token generation
    """
    model.eval()
    src_ids = src_ids.to(device)

    # ── Step 1: Encode source once ──
    enc_out = model.encode(src_ids)

    # ── Step 2: Initialize decoder with BOS token ──
    generated = [bos_token_id]
    kv_caches = None

    # ── Step 3: Autoregressive loop ──
    for step in range(max_new_tokens):
        # Use only the last token as input when KV cache is used
        if kv_caches is not None:
            input_ids = torch.tensor([[generated[-1]]], device=device)
        else:
            input_ids = torch.tensor([generated], device=device)

        # Decoder forward pass
        logits, kv_caches = model.decode_step(input_ids, enc_out, kv_caches)

        # ── Step 4: Pick next token based on strategy ──
        if decoding == "greedy":
            next_token = greedy_decode(logits).item()
        elif decoding == "top_k":
            next_token = top_k_sample(logits, k=top_k, temperature=temperature).item()
        elif decoding == "top_p":
            next_token = top_p_sample(logits, p=top_p, temperature=temperature).item()
        else:
            raise ValueError(f"Unknown decoding strategy: {decoding}")

        generated.append(next_token)

        # ── Step 5: Stop if EOS token generated ──
        if next_token == eos_token_id:
            break

    return generated


# ─────────────────────────────────────────────────────────────
# 10. Demo — Toy Model Inference
# ─────────────────────────────────────────────────────────────

if __name__ == "__main__":
    print("=" * 60)
    print("  Transformer Inference Demo")
    print("=" * 60)

    # Hyperparameters (small model for demo)
    VOCAB_SIZE = 1000
    D_MODEL = 128
    NUM_HEADS = 4
    NUM_LAYERS = 2
    D_FF = 512
    PAD_IDX = 0
    BOS_IDX = 1
    EOS_IDX = 2

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"\nDevice: {device}")

    # ── Build model ──
    model = Transformer(
        vocab_size=VOCAB_SIZE,
        d_model=D_MODEL,
        num_heads=NUM_HEADS,
        num_layers=NUM_LAYERS,
        d_ff=D_FF,
        pad_idx=PAD_IDX,
    ).to(device)

    total_params = sum(p.numel() for p in model.parameters())
    print(f"Model Parameters: {total_params:,}")

    # ── Dummy source sequence ──
    src_ids = torch.randint(3, VOCAB_SIZE, (1, 10)).to(device)   # batch=1, src_len=10
    print(f"\nSource token IDs: {src_ids[0].tolist()}")

    # ── Run inference with different strategies ──
    for strategy in ["greedy", "top_k", "top_p"]:
        output = generate(
            model=model,
            src_ids=src_ids,
            bos_token_id=BOS_IDX,
            eos_token_id=EOS_IDX,
            max_new_tokens=15,
            decoding=strategy,
            temperature=0.8,
            top_k=20,
            top_p=0.9,
            device=device
        )
        print(f"\n[{strategy.upper():6}] Generated tokens: {output}")

    # ── Demonstrate attention manually ──
    print("\n" + "─" * 40)
    print("Manual Attention Demo:")
    B, Sq, Sk, H = 1, 4, 6, 8
    Q = torch.randn(B, H, Sq, 64)
    K = torch.randn(B, H, Sk, 64)
    V = torch.randn(B, H, Sk, 64)
    out, weights = scaled_dot_product_attention(Q, K, V)
    print(f"  Q shape: {Q.shape}")
    print(f"  K shape: {K.shape}")
    print(f"  V shape: {V.shape}")
    print(f"  Attn output shape: {out.shape}")
    print(f"  Attn weights shape: {weights.shape}")
    print(f"  Attn weights sum (should be ~1.0): {weights[0, 0, 0].sum().item():.4f}")

    print("\n✅ Transformer inference pipeline complete!")

  Transformer Inference Demo

Device: cuda
Model Parameters: 1,182,696

Source token IDs: [269, 736, 104, 234, 932, 978, 924, 413, 739, 534]

[GREEDY] Generated tokens: [1, 741, 741, 741, 741, 741, 22, 741, 22, 741, 22, 741, 22, 741, 22, 741]

[TOP_K ] Generated tokens: [1, 710, 741, 943, 932, 21, 269, 21, 206, 22, 242, 932, 932, 651, 639, 57]

[TOP_P ] Generated tokens: [1, 24, 206, 437, 302, 985, 565, 219, 934, 391, 939, 278, 432, 391, 201, 590]

────────────────────────────────────────
Manual Attention Demo:
  Q shape: torch.Size([1, 8, 4, 64])
  K shape: torch.Size([1, 8, 6, 64])
  V shape: torch.Size([1, 8, 6, 64])
  Attn output shape: torch.Size([1, 8, 4, 64])
  Attn weights shape: torch.Size([1, 8, 4, 6])
  Attn weights sum (should be ~1.0): 1.0000

✅ Transformer inference pipeline complete!
